# Data Exploration: MLB Pitchers 2026

This notebook provides initial exploration of Statcast pitch-by-pitch data for three NL ace pitchers:
- **Shohei Ohtani** (Los Angeles Dodgers)
- **Cristopher Sanchez** (Philadelphia Phillies)
- **Jacob Misiorowski** (Milwaukee Brewers)

We will clean the data, examine its structure, and compute summary statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'
PITCHERS = {
    'Ohtani': 'shohei_ohtani.csv',
    'Sanchez': 'cristopher_sanchez.csv',
    'Misiorowski': 'jacob_misiorowski.csv'
}

dfs = {}
for name, fname in PITCHERS.items():
    df = pd.read_csv(f'{DATA_DIR}/{fname}')
    df['player_name_clean'] = df['player_name'].str.replace('\n', ', ', regex=False)
    dfs[name] = df

df_all = pd.concat(dfs.values(), ignore_index=True)
print(f'Total pitches: {len(df_all)}')
print(f'\nPer pitcher:')
print(df_all['player_name_clean'].value_counts())

In [ ]:
# Summary statistics per pitcher
for name, df in dfs.items():
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    print(f'  Pitches: {len(df)}')
    print(f'  Games: {df["game_date"].nunique()}')
    print(f'  Date range: {df["game_date"].min()} to {df["game_date"].max()}')
    print(f'  Pitch types: {sorted(df["pitch_type"].dropna().unique())}')
    vel = df['release_speed'].dropna()
    print(f'  Velocity: {vel.mean():.1f} +/- {vel.std():.1f} mph (max {vel.max():.1f})')
    spin = df['release_spin_rate'].dropna()
    print(f'  Spin rate: {spin.mean():.0f} +/- {spin.std():.0f} rpm')
    in_zone = df['zone'].dropna()
    zone_pct = (in_zone <= 9).sum() / len(in_zone) * 100 if len(in_zone) > 0 else 0
    print(f'  Zone%: {zone_pct:.1f}%')
    whiffs = df[df['description'].str.contains('swinging_strike|swinging_strike_blocked', na=False)]
    swings = df[df['description'].str.contains('swinging_strike|hit_into_play|foul', na=False)]
    whiff_pct = len(whiffs) / len(swings) * 100 if len(swings) > 0 else 0
    print(f'  Whiff%: {whiff_pct:.1f}% ({len(whiffs)}/{len(swings)})')

In [ ]:
# Check for missing data
print('Missing values per key column:')
key_cols = ['release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z',
            'release_extension', 'effective_speed', 'zone',
            'estimated_woba_using_speedangle', 'delta_pitcher_run_exp',
            'api_break_z_with_gravity', 'api_break_x_arm']
for name, df in dfs.items():
    print(f'\n--- {name} ---')
    missing = df[key_cols].isna().sum()
    print(missing[missing > 0])

In [ ]:
# Pitch type distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    pitch_counts = df['pitch_type'].value_counts()
    colors = plt.cm.Set2(np.linspace(0, 1, len(pitch_counts)))
    ax.pie(pitch_counts.values, labels=pitch_counts.index, autopct='%1.1f%%',
           colors=colors, startangle=90)
    ax.set_title(f'{name}\nPitch Arsenal', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/pitch_arsenal_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Velocity distribution by pitch type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    pitch_types = df['pitch_type'].dropna().unique()
    data = [df[df['pitch_type'] == pt]['release_speed'].dropna().values for pt in pitch_types]
    bp = ax.boxplot(data, label=pitch_types, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(pitch_types)))):
        patch.set_facecolor(color)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_ylabel('Release Speed (mph)')
    ax.set_ylim(70, 105)
plt.suptitle('Velocity Distribution by Pitch Type', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/velocity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Horizontal and vertical movement
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, df) in zip(axes, dfs.items()):
    for pt in df['pitch_type'].dropna().unique():
        sub = df[df['pitch_type'] == pt]
        ax.scatter(sub['pfx_x'], sub['pfx_z'], alpha=0.5, s=20, label=pt)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Horizontal Movement (in)')
    ax.set_ylabel('Vertical Movement (in)')
    ax.set_title(f'{name}', fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_xlim(-25, 25)
    ax.set_ylim(-25, 25)
plt.suptitle('Pitch Movement Profiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/pitch_movement.png', dpi=150, bbox_inches='tight')
plt.show()